## Using PyTorch DLC to Host the Whisper Model for Automatic Speech Recognition Tasks

## Common set up 

**❗If you run this notebook in SageMaker Studio, please select the SageMaker Distribution 4.x image and choose the ml.m5.large instance.**

In [1]:
# Install required packages (setuptools<71 needed for pkg_resources on Python 3.12)
# NOTE: First run takes 3-5 minutes (whisper builds from source). Subsequent runs are instant.
# INSTRUCTOR: Pre-run this cell before the workshop starts.
%pip install 'setuptools<71' openai-whisper==20231117 -q
%pip install torchaudio==2.5.1 -q
%pip install datasets==3.2.0 -q
%pip install sagemaker==2.232.1 -q
%pip install librosa==0.10.2 -q
%pip install soundfile==0.12.1 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
sagemaker-serve 1.11.0 requires onnxruntime, which is not installed.
autogluon-multimodal 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.3.1 which is incompatible.
autogluon-timeseries 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.3.1 which is incompatible.
sagemaker-serve 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
torchaudio 2.5.1 requires torch==2.5.1, but you have torch 2.3.1 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
sagemaker-serve 1.11.0 requires onnxruntime, which is not installed.
autogluon-multimodal 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.5.1 which is incompatible.
autogluon-timeseries 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.5.1 which is incompatible.
sagemaker-serve 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
openai-whisper 20231117 requires triton<3,>=2.0.0, but you have triton 3.1.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
strands-agents-tools 0.1.9 requires dill<0.5.0,>=0.4.0, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.2.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 3.2.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


**❗Please restart the kernel before executing the cells below.**

In [2]:
# Import required packages
import torch
import whisper
import torchaudio
import sagemaker
import time
import json
import boto3
import soundfile as sf
from datasets import load_dataset

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Basic configurations
sess = sagemaker.session.Session()
bucket = sess.default_bucket()
prefix = 'whisper_blog_post'
role = sagemaker.get_execution_role()

# below boto3 clients are for invoking asynchronous endpoint 
sm_runtime = boto3.client("sagemaker-runtime")

### Create Whisper pytorch model artifacts and upload to S3 bucket

In [4]:
# Load the PyTorch model and save it in the local repo
model = whisper.load_model("base")
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'dims': model.dims.__dict__,
    },
    'base.pt'
)

/opt/conda/lib/python3.12/site-packages/whisper/__init__.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


In [5]:
# Move the model to the 'model' directory and create a tarball
!mkdir -p model
!mv base.pt model
!tar cvzf model.tar.gz -C model/ .

# Upload the model to S3
model_uri = sess.upload_data('model.tar.gz', bucket=bucket, key_prefix=f"{prefix}/pytorch/model")
!rm model.tar.gz
!rm -rf model
model_uri

./
./base.pt


's3://sagemaker-us-west-2-149057604171/whisper_blog_post/pytorch/model/model.tar.gz'

In [6]:
# Generate a unique model name and provide image uri

id = int(time.time())
model_name = f'whisper-pytorch-model-{id}'

# Resolve region automatically
region = boto3.Session().region_name
image = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-inference:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"

In [7]:
# Create a PyTorchModel for deployment
from sagemaker.pytorch.model import PyTorchModel

whisper_pytorch_model = PyTorchModel(
    model_data=model_uri,
    image_uri=image,
    role=role,
    entry_point="inference.py",
    source_dir='code_sagemaker_4',
    name=model_name,
    env = {
        'MMS_MAX_REQUEST_SIZE': '2000000000',
        'MMS_MAX_RESPONSE_SIZE': '2000000000',
        'MMS_DEFAULT_RESPONSE_TIMEOUT': '900' 
    } # MMS env variables for the HuggingFace DLC
)

### Real-time inference 

In [8]:
from sagemaker.serializers import DataSerializer
from sagemaker.deserializers import JSONDeserializer

# Define serializers and deserializer
audio_serializer = DataSerializer(content_type="audio/x-audio")
deserializer = JSONDeserializer()

In [9]:
%%time
# Deploy the model for real-time inference
endpoint_name = f'whisper-pytorch-real-time-endpoint-{id}'

real_time_predictor = whisper_pytorch_model.deploy(
    initial_instance_count=1,  # number of instances
    instance_type="ml.g6.xlarge",  # instance type
    endpoint_name = endpoint_name,
    serializer=audio_serializer,
    deserializer = deserializer
)
# this step takes about 7 mins

-

-

-

-

-

-

-

-

!

CPU times: user 1min 26s, sys: 1.28 s, total: 1min 27s
Wall time: 5min 59s


In [10]:
# Download a test data sample from huggingface dataset
dataset = load_dataset('MLCommons/peoples_speech', 'clean', split='train', streaming=True)
sample = next(iter(dataset))
audio_data = sample['audio']['array']
output_path = 'sample_audio.wav'
sf.write(output_path, audio_data, sample['audio']['sampling_rate'])

print(f"Audio sample saved to '{output_path}'.") 

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Audio sample saved to 'sample_audio.wav'.


In [11]:
import json
# Perform real-time inference
audio_path = "sample_audio.wav" 
response = real_time_predictor.predict(data=audio_path)

print(json.loads(response[0])['text'])

 I wanted to just share a few things, but I'm not going to not share as much as I wanted to share because we are starting late. I'd like to get this thing going so we all get home at a decent hour. This election is very important to us.


In [12]:
# optional: Delete real-time inference endpoint, this is not required for below steps
real_time_predictor.delete_endpoint()


### Batch Transform Inference

In [13]:
# Create a transformer
whisper_transformer = whisper_pytorch_model.transformer(
    instance_count = 1,
    instance_type = "ml.g6.xlarge", 
    output_path="s3://{}/{}/batch-transform/".format(bucket, prefix),
    max_payload = 100
)

Using already existing model: whisper-pytorch-model-1781166523


In [14]:
# Use the same S3 bucket for batch transform input

data = f"s3://{bucket}/{prefix}/audio-files/"

In [15]:
# Define request data and job name
job_name = f"whisper-pytorch-batch-transform-{id}"

# Start batch transform job
whisper_transformer.transform(data = data, job_name= job_name, wait = False)

INFO:sagemaker:Creating transform job with name: whisper-pytorch-batch-transform-1781166523


ResourceLimitExceeded: An error occurred (ResourceLimitExceeded) when calling the CreateTransformJob operation: The account-level service limit 'ml.g6.xlarge for transform job usage' is 0 Instances, with current utilization of 0 Instances and a request delta of 1 Instances. Please use AWS Service Quotas to request an increase for this quota. If AWS Service Quotas is not available, contact AWS support to request an increase for this quota.

### Asynchronous Inference 

In [ ]:
%%time
from sagemaker.async_inference import AsyncInferenceConfig

# Create an AsyncInferenceConfig object
async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/{prefix}/output", 
    max_concurrent_invocations_per_instance = 4,
    # notification_config = {
            #   "SuccessTopic": "arn:aws:sns:us-east-2:123456789012:MyTopic",
            #   "ErrorTopic": "arn:aws:sns:us-east-2:123456789012:MyTopic",
    # }, #  Notification configuration 
)

# Deploy the model for async inference
endpoint_name = f'whisper-pytorch-async-endpoint-{id}'
async_predictor = whisper_pytorch_model.deploy(
    async_inference_config=async_config,
    initial_instance_count=1, # number of instances
    instance_type ='ml.g6.xlarge', # instance type
    endpoint_name = endpoint_name
)

In [ ]:
# Use the same S3 bucket for async inference input

input_path = f"s3://{bucket}/{prefix}/audio-files/sample_audio.wav"

In [ ]:
# Perform async inference
initial_args = {'ContentType':"audio/x-audio"}
response = async_predictor.predict_async(initial_args = initial_args, input_path=input_path)
response.output_path

### Optional: Test autoscaling configurations for Async inference 

In [ ]:
autoscale = boto3.client('application-autoscaling') 
resource_id='endpoint/' + endpoint_name + '/variant/' + 'AllTraffic'

# Register scalable target
register_response = autoscale.register_scalable_target(
    ServiceNamespace='sagemaker', 
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=0,  
    MaxCapacity=3 # * check how many instances available in your account
)

# Define scaling policy
scalingPolicy_response = autoscale.put_scaling_policy(
    PolicyName='Invocations-ScalingPolicy',
    ServiceNamespace='sagemaker', # The namespace of the AWS service that provides the resource. 
    ResourceId=resource_id,  
    ScalableDimension='sagemaker:variant:DesiredInstanceCount', # SageMaker supports only Instance Count
    PolicyType='TargetTrackingScaling', # 'StepScaling'|'TargetTrackingScaling'
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 3.0, # The target value for the metric. 
        'CustomizedMetricSpecification': {
            'MetricName': 'ApproximateBacklogSizePerInstance',
            'Namespace': 'AWS/SageMaker',
            'Dimensions': [
                {'Name': 'EndpointName', 'Value': endpoint_name }
            ],
            'Statistic': 'Average',
        },
        'ScaleInCooldown': 60, # The cooldown period helps you prevent your Auto Scaling group from launching or terminating 
                                # additional instances before the effects of previous activities are visible. 
                                # You can configure the length of time based on your instance startup time or other application needs.
                                # ScaleInCooldown - The amount of time, in seconds, after a scale in activity completes before another scale in activity can start. 
        'ScaleOutCooldown': 60 # ScaleOutCooldown - The amount of time, in seconds, after a scale out activity completes before another scale out activity can start.
        
        # 'DisableScaleIn': True|False - indicates whether scale in by the target tracking policy is disabled. 
                            # If the value is true , scale in is disabled and the target tracking policy won't remove capacity from the scalable resource.
    }
)

scalingPolicy_response

In [ ]:
# Trigger 1000 asynchronous invocations with autoscaling from 1 to 3
# then scale down to 0 on completion

print(endpoint_name)
for i in range(1,1000):
    response = sm_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name, 
    InputLocation=input_path)
    
print("\nAsync invocations for PyTorch serving with autoscaling\n")

### Clean up

In [ ]:
# Delete Asynchronous inference endpoint
async_predictor.delete_endpoint()